# 70a · Bước 0 — chọn bộ nén, và Bước 1 — chuẩn bị dữ liệu

Notebook đầu tiên của hướng **sinh ảnh bằng sửa ảnh thật**. Kế hoạch đầy đủ ở
`docs/PLAN_SINH_ANH_IMG2IMG.md`.

**Vòng này trả lời đúng một câu:** bộ nén của latent diffusion có giữ được bệnh lý trên X-quang
không, và nếu có thì bộ nén nào. Chưa sinh ảnh nào cả.

```
0a  vong nen        -> bo nen giu duoc benh ly den dau, THEO TUNG NHAN
0c  thich nghi      -> UNet + LoRA hoc tren latent do den dau   (cell sau, chua co trong ban nay)
```

## ⚠ Notebook này KHÔNG ghi gì vào `stage4_nb68b`

`C0` của `68b` gọi `ensure_res()` ở mức module, tức ghi `_probe.txt` thẳng vào
`stage4_nb68b/r1022_E8` — thư mục đang giữ kết quả `s43`. Nên `C0` ở đây **viết mới**: ghi vào
`stage4_nb70a`, và chỉ **đọc** checkpoint `w_s*_ep*.pt` của `68b`. Có cổng chặn ở `put()`.

`C1`–`C4b` copy **verbatim** từ `68b` (đã quét: không thao tác ghi nào lên Drive).

## C0 · Config — thư mục riêng, chỉ đọc kết quả của `68b`

In [1]:
import os, json, math, time, re, glob, datetime, shutil, tarfile, io as _io
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
try:
    import timm
except ImportError:
    import subprocess, sys as _s
    subprocess.run([_s.executable, '-m', 'pip', 'install', '-q', 'timm']); import timm
try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = os.environ.get('CXRLT_DATA', '')
    assert DATA_DIR, 'khong phai Colab va chua dat CXRLT_DATA -> gan thang duong dan `planB_out`'
OUT_DIR = os.path.join(DATA_DIR, 'cxrlt_2024'); EMB_DIR = os.path.join(OUT_DIR, 'embeddings')

# Chan spam `Exception ignored in __del__: can only test a child process` cua DataLoader.
import sys as _sys
_old_unraisable = _sys.unraisablehook
def _quiet_unraisable(u):
    if isinstance(u.exc_value, AssertionError) and 'child process' in str(u.exc_value): return
    _old_unraisable(u)
_sys.unraisablehook = _quiet_unraisable

SIZE_SHARD = 1022
SIZE, BATCH, ACCUM = 1022, 12, 24     # BATCH/ACCUM giu de C1-C4b khoi thieu bien; 70a KHONG train
CHE_DO_THU = False
print(f'DO PHAN GIAI {SIZE}')

# ==== THU MUC: 70a ghi vao CHO CUA RIENG NO. 68b chi de DOC. ====
RES_LOC = '/content/nb70a'; os.makedirs(RES_LOC, exist_ok=True)
RES41D  = os.path.join(OUT_DIR, 'stage4_nb70a')      # giu TEN BIEN de C1-C4b khoi phai sua
RES41C  = os.path.join(OUT_DIR, 'stage4_nb41c')
CAM_GHI = os.path.join(OUT_DIR, 'stage4_nb68b')      # tuyet doi khong ghi vao day


def _dirs_68b(goc='stage4_nb68b', run='r1022_E8'):
    """Drive cho phep thu muc TRUNG TEN. Checkpoint cua 68b nam rai o 'r1022_E8',
       'r1022_E8 (1)', ... -> phai duyet HET, neu khong `find()` chi thay mot ban."""
    out = []
    if not os.path.isdir(OUT_DIR): return out
    for x in sorted(os.listdir(OUT_DIR)):
        if not (x == goc or x.startswith(goc + ' (')): continue
        b = os.path.join(OUT_DIR, x)
        for y in sorted(os.listdir(b)):
            if (y == run or y.startswith(run + ' (')) and os.path.isdir(os.path.join(b, y)):
                out.append(os.path.join(b, y))
    return out


DIRS_68B = _dirs_68b()
SEARCH = [RES_LOC, RES41D] + DIRS_68B                # DOC tu 68b, GHI vao 70a
print(f'doc checkpoint 68b tu {len(DIRS_68B)} thu muc: '
      + ', '.join(os.path.basename(d) for d in DIRS_68B[:6])
      + (' ...' if len(DIRS_68B) > 6 else ''))
assert DIRS_68B, ('khong thay stage4_nb68b/r1022_E8* -> 0a khong co checkpoint de cham diem. '
                  'Kiem thu muc trung ten tren Drive truoc khi chay tiep.')

_RES_OK = False
def ensure_res():
    """Tao DUNG MOT LAN moi session, xac minh bang GHI THAT + DOC LAI (`isdir` tren FUSE
       khong dang tin), roi khong bao gio goi `makedirs` nua. Goi makedirs moi file la
       cach du an nay da de ra 17 thu muc trung ten."""
    global _RES_OK
    if _RES_OK: return RES41D
    if not os.path.isdir(RES41D):
        os.makedirs(RES41D, exist_ok=True)
    # GIU lai file nay, KHONG xoa. Drive khong giu thu muc RONG dang tin qua FUSE:
    # 15/09 C0 bao tao thanh cong, mot gio sau shutil.copy vao dung cho do lai
    # FileNotFoundError vi thu muc rong da bien mat.
    _t = os.path.join(RES41D, '_tao_boi_70a.txt')
    with open(_t, 'w') as _f: _f.write('70a · Buoc 0. Dung xoa: thu muc rong se bien mat.\n')
    assert open(_t).read().startswith('70a'), f'{RES41D} ghi-doc lai khong duoc -> mount hong'
    _RES_OK = True
    print(f'thu muc ket qua tren Drive: {RES41D}  (tao 1 lan, da xac minh)')
    return RES41D


PUT_FAIL = []
def put(name, fn, kiem_byte=True):
    """Ghi LOCAL truoc (chac chan), roi sao len Drive va XAC MINH.

       `kiem_byte=True`: so TUNG BYTE, khong phai kich thuoc. Ly do (68b C0 da ghi):
       ghi de cung ten qua Drive FUSE thi chi lan ghi DAU xuong dia, ma hai lan chay
       cho file kich thuoc gan bang nhau -> kiem kich thuoc KHONG bat duoc.
       Dat False cho file lon (.npy) vi phai doc lai het."""
    lp = os.path.join(RES_LOC, name); fn(lp)
    dp = os.path.join(RES41D, name)
    assert not os.path.exists(dp), (
        f'{name} DA CO tren Drive. Khong ghi de: qua FUSE thi lan ghi thu hai co the '
        'khong xuong dia ma van bao thanh cong. Doi ten file (co dau thoi gian) '
        'hoac xoa tay ban cu neu chac chan khong can.')
    assert os.path.abspath(dp).startswith(os.path.abspath(RES41D)), 'duong dan ghi ra ngoai 70a'
    assert not os.path.abspath(dp).startswith(os.path.abspath(CAM_GHI)), \
        'CHAN: dang dinh ghi vao stage4_nb68b — do la thu muc ket qua cua 68b, chi duoc DOC'
    global _RES_OK
    _loi = ''
    for _lan in range(3):
        try:
            ensure_res(); shutil.copy(lp, dp)
            if os.path.getsize(dp) != os.path.getsize(lp): raise OSError('kich thuoc lech')
            if kiem_byte and open(dp, 'rb').read() != open(lp, 'rb').read():
                raise OSError('noi dung tren Drive KHAC ban local (ghi khong xuong dia)')
            return lp
        except Exception as e:
            _loi = f'{type(e).__name__}: {e}'
            if isinstance(e, FileNotFoundError):
                _RES_OK = False        # thu muc da bien mat -> cho ensure_res tao lai
            time.sleep(2 * (_lan + 1))
    PUT_FAIL.append(name); print(f'  !! sao {name} len Drive THAT BAI sau 3 lan ({_loi})')
    return lp


def drive_gate():
    """Goi CUOI cell co ghi. Khong goi thi ket qua chi con o /content va chet theo session."""
    if not PUT_FAIL: return
    raise AssertionError(
        f'{len(PUT_FAIL)} file KHONG len duoc Drive: {PUT_FAIL[:4]}\n'
        f'  Hien vat CON o {RES_LOC} — dung tat session truoc khi cuu.\n'
        f'  Cuu: !cp -r {RES_LOC} /content/drive/MyDrive/_cuu70a')


def find(pat):
    seen = {}
    for d in SEARCH:
        for f in sorted(glob.glob(os.path.join(d, pat))): seen.setdefault(os.path.basename(f), f)
    return [seen[k] for k in sorted(seen)]


ensure_res()
LOCAL = f'/content/cxrlt_{SIZE}'
READY_X = os.path.join(OUT_DIR, 'cxrlt_518_ready')
SRC_PIPE = [os.path.join(OUT_DIR, 'stage4_nb57', 'r3'), os.path.join(OUT_DIR, 'stage4_nb53', 'r3')]
ARM_PIPE = ('tpb', 't1_prior_both'); ARM_IMG = 'mldec'

MODEL = 'convnextv2_nano.fcmae_ft_in22k_in1k'
P_BB  = os.path.join(RES41C, 'convnextv2_padchest.pt')
P_OOF = os.path.join(OUT_DIR, 'stage4_nb52a', 'xgb_oof.npy')
SEED, SEEDS = 42, (42, 43)
EPOCHS = 8
LR, WD, WARMUP, CLIP = 1e-4, 0.05, 0.05, 5.0
_MB_ANH  = 3 * SIZE * SIZE * 4 / 1e6
NW       = min(4, (os.cpu_count() or 2))
PREFETCH = 2
BS_EVAL  = 32
BS_DIAG  = 16
D_MLD, D_ETOK, EHR_CLIP = 512, 256, 10.0
MOC, GATE_BLEND = 0.8236, 0.005
DO_TEST = False                                   # 70a KHONG cham test

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
assert dev == 'cuda', 'KHONG thay GPU'
AMP = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
print(f'thiet bi: {torch.cuda.get_device_name(0)} | '
      f'VRAM {torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB | AMP {AMP}')

_bb_loc = '/content/convnextv2_padchest.pt'
BB_USE = P_BB if os.path.exists(P_BB) else (_bb_loc if os.path.exists(_bb_loc) else None)
assert BB_USE, f'khong thay trong so PadChest o CA {P_BB} lan {_bb_loc} -> chay 41c C3 truoc'
for p, hd in ((P_OOF, '52a'), (os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet'), '29c'),
              (os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json'), '29c')):
    assert os.path.exists(p), f'khong thay {p} -> chay {hd} truoc'

INV = json.load(open(os.path.join(OUT_DIR, 'cxrlt_inventory.json')))
LAB40 = INV['labels_task1']; C = len(LAB40)
TIER_CU = {t: np.array([LAB40.index(l) for l in INV['tier'][t]]) for t in ('head','medium','tail')}
ORDER = pd.read_parquet(os.path.join(EMB_DIR, 'emb_index.parquet'))
YM = (pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_index.parquet'), columns=['dicom_id'] + LAB40)
        .set_index('dicom_id').loc[ORDER.dicom_id].reset_index())[LAB40].to_numpy('float32')
SPLIT = ORDER.split.to_numpy(); SUB = ORDER.subject_id.to_numpy()
TR, VA = SPLIT == 'train', SPLIT == 'val'; TE = ~(TR | VA)
_prev = YM[SPLIT == 'train'].mean(0)
TIER = {'head':   np.where(_prev > 0.10)[0],
        'medium': np.where((_prev >= 0.01) & (_prev <= 0.10))[0],
        'tail':   np.where(_prev < 0.01)[0]}
print('TIER = CHINH THUC ' + str({k: len(v) for k, v in TIER.items()}))
# CHI CO BA TAP. Khong con lat chon / lat khop / tap con nao khac (Tuan, 15/09).
# 68b mang theo TF_ (ban sao cua TR) va SL (lat chon 15%) — da go het.
assert not (set(SUB[TR]) & set(SUB[VA])), 'benh nhan train ro ri sang VAL'
assert not (set(SUB[TR]) & set(SUB[TE])), 'benh nhan train ro ri sang TEST'
assert not (set(SUB[VA]) & set(SUB[TE])), 'benh nhan val ro ri sang TEST'
I_TR, I_VA, I_TE = (np.where(m)[0] for m in (TR, VA, TE))
assert len(I_TR) + len(I_VA) + len(I_TE) == len(ORDER), \
    'ba tap khong phu het — con mot tap thu tu o dau do'
print(f'{len(ORDER):,} anh = train {len(I_TR):,} + val {len(I_VA):,} + test {len(I_TE):,}')


Mounted at /content/drive
DO PHAN GIAI 1022
doc checkpoint 68b tu 1 thu muc: r1022_E8
thu muc ket qua tren Drive: /content/drive/MyDrive/MIMIC_CXR/planB_out/cxrlt_2024/stage4_nb70a  (tao 1 lan, da xac minh)
thiet bi: NVIDIA RTX PRO 6000 Blackwell Server Edition | VRAM 102 GB | AMP torch.bfloat16
TIER = CHINH THUC {'head': 9, 'medium': 15, 'tail': 16}
377,110 anh | train 258,871 | val 39,293 | test 78,946


## C1 · EHR + phim trước + ảnh + dự đoán đường ống

In [2]:
SPEC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_ehr_spec.json')))
E_DF = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_ehr.parquet'))
_sid = ORDER.study_id.to_numpy()
COLS = [c for c in SPEC['Q_FULL'] if c in E_DF.columns]
_m = np.nan_to_num((E_DF.set_index('study_id')[COLS].reindex(_sid)
                    .apply(pd.to_numeric, errors='coerce').to_numpy('float32')), nan=0.0)
# Cot NHI PHAN thi KHONG z-score: p = 1/219,785 -> gia tri chuan hoa 468 -> softmax bao hoa -> NaN
_b = np.array([np.isin(np.unique(_m[:, j]), (0.0, 1.0)).all() for j in range(_m.shape[1])])
_mu = _m[TR].mean(0, keepdims=True); _sd = _m[TR].std(0, keepdims=True) + 1e-6
_mu[:, _b] = 0.0; _sd[:, _b] = 1.0
E_ALL = np.clip((_m - _mu) / _sd, -EHR_CLIP, EHR_CLIP).astype('float32')
_keep = [i for i, c in enumerate(COLS) if not re.match(r'^micro_org_', c)]
assert len(COLS) - len(_keep) == 30, f'bo {len(COLS)-len(_keep)} cot micro_org_*, phai la 30'
EHR_COL = E_ALL[:, _keep]

_MIC = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_micro_full.parquet')).set_index('study_id')
_VOC = json.load(open(os.path.join(OUT_DIR, 'cxrlt_micro_vocab.json')))
V_MIC = int(_VOC['V']) + 1; PAD_ID = V_MIC - 1
_mic = _MIC.reindex(_sid)
_ids = _mic.micro_org_ids.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
_cnt = _mic.micro_org_cnt.apply(lambda v: list(v) if isinstance(v, (list, np.ndarray)) else [])
MIC_L = max(1, int(_ids.map(len).max())); assert MIC_L <= 64
MIDS = np.full((len(ORDER), MIC_L), PAD_ID, 'int64'); MCNT = np.zeros((len(ORDER), MIC_L), 'float32')
for r, (a, b) in enumerate(zip(_ids, _cnt)):
    if a: MIDS[r, :len(a)] = a; MCNT[r, :len(a)] = b

XGF = np.load(P_OOF).astype('float32')
assert XGF.shape == (len(ORDER), C) and np.isfinite(XGF).all()
_l = np.log(np.clip(XGF, 1e-6, 1-1e-6) / (1 - np.clip(XGF, 1e-6, 1-1e-6)))
XGB_F = np.clip((_l - _l[TR].mean(0, keepdims=True)) / (_l[TR].std(0, keepdims=True) + 1e-6),
                -EHR_CLIP, EHR_CLIP).astype('float32')
FEAT = np.concatenate([EHR_COL, XGB_F], 1)
print(f'FEAT {FEAT.shape} | tui vi khuan V={V_MIC-1} dai {MIC_L}')

_BASE = pd.read_parquet(os.path.join(OUT_DIR, 'cxrlt_base.parquet'))
_BASE['index_time'] = pd.to_datetime(_BASE.index_time)
_B = (_BASE[_BASE.study_id.isin(set(ORDER.study_id.unique()))]
      .sort_values(['subject_id','index_time']).reset_index(drop=True))
_B['prev_study'] = _B.groupby('subject_id', sort=False).study_id.shift(1)
_PV = dict(zip(_B.study_id.to_numpy(), _B.prev_study.to_numpy()))
_rw = {}
for _i, _s in enumerate(_sid): _rw.setdefault(_s, []).append(_i)
PREV_ROW = np.full(len(ORDER), -1, 'int64')
for _i, _s in enumerate(_sid):
    _p2 = _PV.get(_s, np.nan)
    if not (isinstance(_p2, float) and np.isnan(_p2)): PREV_ROW[_i] = _rw[int(_p2)][0]
_hp = PREV_ROW >= 0
print(f'anh co phim truoc: {_hp.sum():,} / {len(ORDER):,} ({_hp.mean():.1%})  (44a do 67.2%)')
assert _hp.mean() > 0.5 and (PREV_ROW[_hp] != np.arange(len(ORDER))[_hp]).all()

from PIL import Image
os.makedirs(LOCAL, exist_ok=True)
_n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
print(f'anh dang co: {_n:,} / {len(ORDER):,}', flush=True)
from concurrent.futures import ThreadPoolExecutor
# Kho `cxrlt_518_ready` CHI co ban 518 -> o do phan giai khac phai stage tu shard GOC.
if SIZE == 518 and _n < len(ORDER) * 0.999 and os.path.isdir(READY_X):
    _kt = sorted(x for x in os.listdir(READY_X) if x.endswith('.tar'))
    print(f'kho co {len(_kt)} tar -> hydrate', flush=True)
    for _i, f in enumerate(_kt):
        src = os.path.join(READY_X, f); tmp = os.path.join('/content', f)
        try:
            shutil.copy(src, tmp)                # copy TRUOC — khong mo tar tren mount FUSE
            assert os.path.getsize(tmp) == os.path.getsize(src), 'copy NGAN tu Drive'
            with tarfile.open(tmp) as tar: tar.extractall(LOCAL, filter='data')
        except Exception as e:
            print(f'  {f} HONG ({type(e).__name__}) -> bo qua')
        if os.path.exists(tmp): os.remove(tmp)
        if (_i + 1) % 50 == 0: print(f'  hydrate {_i+1}/{len(_kt)}', flush=True)
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
if _n < len(ORDER) * 0.999:
    SHARD_DIR = os.path.join(DATA_DIR, 'cxr_1022_letterbox_cxrlt')
    assert os.path.isdir(SHARD_DIR), f'khong thay CA {READY_X} lan {SHARD_DIR}'
    TARS = sorted(os.path.join(SHARD_DIR, f) for f in os.listdir(SHARD_DIR)
                  if f.endswith('.tar') and f.startswith('lb_'))
    DONE = os.path.join(LOCAL, '_done.json')
    _done = set(json.load(open(DONE))) if os.path.exists(DONE) else set()
    def _one(pl):
        nm, blob = pl; dst = os.path.join(LOCAL, os.path.basename(nm))
        if os.path.exists(dst): return
        try:
            if SIZE == SIZE_SHARD:     # shard DA dung kich thuoc -> ghi THANG, khoi giai ma
                tmp = dst + '.tmp'     # + ma hoa lai 377k anh (tiet kiem PHAN LON thoi gian)
                with open(tmp, 'wb') as _f: _f.write(blob)
                os.replace(tmp, dst); return
            im = Image.open(_io.BytesIO(blob)).convert('L')
            if im.size != (SIZE, SIZE): im = im.resize((SIZE, SIZE), Image.BICUBIC)
        except Exception:
            return                     # mot anh hong KHONG duoc giet ca lan chay
        tmp = dst + '.tmp'; im.save(tmp, 'JPEG', quality=95); os.replace(tmp, dst)   # NGUYEN TU
    with ThreadPoolExecutor(8) as pool:
        for tp in tqdm([t for t in TARS if os.path.basename(t) not in _done],
                       desc=f'stage {SIZE}', unit='tar'):
            with open(tp, 'rb') as f: buf = f.read()   # doc BULK; lap tren mount FUSE se TREO
            assert len(buf) == os.path.getsize(tp), f'{os.path.basename(tp)}: doc NGAN tren mount'
            with tarfile.open(fileobj=_io.BytesIO(buf)) as tar:
                jobs = [(m.name, tar.extractfile(m).read())
                        for m in tar.getmembers() if m.name.endswith('.jpg')]
            list(pool.map(_one, jobs)); del buf, jobs
            _done.add(os.path.basename(tp)); json.dump(sorted(_done), open(DONE, 'w'))
    _n = len([f for f in os.listdir(LOCAL) if f.endswith('.jpg')])
assert _n >= len(ORDER) * 0.999, f'thieu {len(ORDER)-_n:,} anh'
print(f'anh local: {_n:,} / {len(ORDER):,}')

# 64b KHONG nap du doan duong ong nua. Cell blend cua 41e da bi bo: no quet trong so `w`
# TREN LAT CHON, ma o R2a lat chon la du lieu huan luyen -> moi diem tren duong cong deu
# la TRONG MAU. Viec tron lam o `61`, voi `w` DA KHOA (head 0.40 | medium 0.40 | tail 0),
# sau khi ca `64` va `64b` chay xong.


FEAT (377110, 257) | tui vi khuan V=305 dai 10
anh co phim truoc: 253,283 / 377,110 (67.2%)  (44a do 67.2%)
anh dang co: 0 / 377,110


stage 1022:   0%|          | 0/378 [00:00<?, ?tar/s]

anh local: 377,110 / 377,110


## C2 · Kiến trúc — `Tier1`/`Tier1Img`/ML-Decoder verbatim `53` r3, backbone ConvNeXt

`convnextv2_nano` @518 cho `17×17 = 289` token dim 640. `forward_features` trả `[B, H, W, Cf]`
(channels-last ở ConvNeXt của `timm`) — cổng 1 kiểm hình dạng thay vì tin giả định.

In [3]:
class GroupFC(nn.Module):
    def __init__(s, C, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(C, d) * d ** -0.5); s.b = nn.Parameter(torch.zeros(C))
    def forward(s, h): return (h * s.w).sum(-1) + s.b

class XAttn(nn.Module):
    def __init__(s, d, nhead=8, ff=2048, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n1, s.n2 = nn.LayerNorm(d), nn.LayerNorm(d)
        s.ff = nn.Sequential(nn.Linear(d, ff), nn.GELU(), nn.Dropout(drop), nn.Linear(ff, d))
    def forward(s, q, kv, need_w=False):
        a, w = s.at(q, kv, kv, need_weights=need_w, average_attn_weights=True)
        h = s.n1(q + a); return s.n2(h + s.ff(h)), w

class MicroBag(nn.Module):
    def __init__(s, V, d):
        super().__init__()
        s.w = nn.Parameter(torch.randn(V, d) * d ** -0.5)
        s.mlp = nn.Sequential(nn.GELU(), nn.Linear(d, d))
    def forward(s, ids, w): return s.mlp(F.embedding_bag(ids, s.w, per_sample_weights=w, mode='sum'))

class ColTokBag(nn.Module):
    def __init__(s, d_in, d=256, V=None):
        super().__init__()
        s.W = nn.Parameter(torch.randn(d_in, d) * d ** -0.5)
        s.b = nn.Parameter(torch.zeros(d_in, d))
        s.joint = nn.Sequential(nn.Linear(d_in, d), nn.GELU(), nn.Linear(d, d))
        s.ln = nn.LayerNorm(d); s.mb = MicroBag(V, d) if V else None
        if s.mb is not None: s.bln = nn.LayerNorm(d)
    def forward(s, e, mids=None, mcnt=None):
        col = s.b.unsqueeze(0) + e.unsqueeze(-1) * s.W.unsqueeze(0)
        base = s.ln(torch.cat([col, s.joint(e).unsqueeze(1)], 1))
        if s.mb is None: return base
        return torch.cat([base, s.bln(s.mb(mids, mcnt)).unsqueeze(1)], 1)

class Tier1(nn.Module):
    '''Q = patch, K/V = token EHR. `pr` zero-init; `ln_a` chuan hoa DAU VAO cua phep chieu (38d).'''
    def __init__(s, d, nhead=8, drop=0.1):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.n = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d); s.pr = nn.Linear(d, d)
        nn.init.zeros_(s.pr.weight); nn.init.zeros_(s.pr.bias)
    def forward(s, patch, etok):
        a, _ = s.at(s.n(patch), etok, etok, need_weights=False)
        return patch + s.pr(s.ln_a(a))

class Tier1Img(nn.Module):
    '''Ban r3: LayerScale `gamma` 1e-4 — chan cu sup ma zero-init khong chan duoc (0/16 epoch-arm
       o 57, truoc do sup 3 lan lien tiep).'''
    def __init__(s, d, nhead=8, drop=0.1, diff=True):
        super().__init__()
        s.at = nn.MultiheadAttention(d, nhead, dropout=drop, batch_first=True)
        s.nq = nn.LayerNorm(d); s.nk = nn.LayerNorm(d); s.ln_a = nn.LayerNorm(d)
        s.pr = nn.Linear(d, d); s.gamma = nn.Parameter(torch.full((d,), 1e-4)); s.diff = diff
    def forward(s, patch, prior, has):
        a, _ = s.at(s.nq(patch), s.nk(prior), s.nk(prior), need_weights=False)
        z = s.gamma * s.pr(s.ln_a(patch - a if s.diff else a))
        return patch + z * has.view(-1, 1, 1)

class MLDecoder(nn.Module):
    def __init__(s, C, d_in, d=512, nhead=8, ff=1024, drop=0.1, d_ehr_in=0, t1=False):
        super().__init__()
        s.q = nn.Embedding(C, d); s.q.weight.requires_grad_(False)   # query lop KHONG hoc (bai goc)
        s.pin = nn.Linear(d_in, d); s.img = XAttn(d, nhead, ff, drop)
        s.t1 = Tier1(d, nhead, drop) if t1 else None
        if t1: s.ein = nn.Linear(d_ehr_in, d)
        s.out = GroupFC(C, d)
    def forward(s, patch, etok=None):
        q = s.q.weight.unsqueeze(0).expand(patch.shape[0], -1, -1)
        kv = s.pin(patch)
        if s.t1 is not None: kv = s.t1(kv, s.ein(etok))
        h, _ = s.img(q, kv); return s.out(h)

class ASL(nn.Module):
    '''Verbatim nb11..59. gn=4, gp=0 la cau hinh 35 da chot.'''
    def __init__(s, gn=4, gp=0, clip=0.05, eps=1e-8):
        super().__init__(); s.gn, s.gp, s.clip, s.eps = gn, gp, clip, eps
    def forward(s, x, y):
        y = y.float(); xp = torch.sigmoid(x); xn = 1 - xp
        if s.clip: xn = (xn + s.clip).clamp(max=1)
        lo = y*torch.log(xp.clamp(min=s.eps)) + (1-y)*torch.log(xn.clamp(min=s.eps))
        pt = xp*y + xn*(1-y); g = s.gp*y + s.gn*(1-y)
        return -(lo * torch.pow(1-pt, g)).mean()

from sklearn.metrics import average_precision_score
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
mac = lambda Y_, P_: float(np.nanmean(ap_cols(Y_, P_)))
tie = lambda Y_, P_: {t: float(np.nanmean(ap_cols(Y_, P_)[i])) for t, i in TIER.items()}
_r6 = np.random.default_rng(6); _y6 = (_r6.random((3000, C)) < 0.2).astype('float32')
_s6 = _r6.random((3000, C)).astype('float32')
assert np.nanmax(np.abs(ap_cols(_y6, _s6) - np.array(
    [average_precision_score(_y6[:, j], _s6[:, j]) for j in range(C)]))) < 1e-6
print('cong 1: ap_cols khop sklearn')

_m0 = timm.create_model(MODEL, pretrained=True, num_classes=0, in_chans=1)
_cfg = timm.data.resolve_model_data_config(_m0)
MU, SD = float(np.mean(_cfg['mean'])), float(np.mean(_cfg['std']))
_IN_SD = {k: v.clone() for k, v in _m0.state_dict().items()}; del _m0

def build_backbone():
    bb = timm.create_model(MODEL, pretrained=False, num_classes=0, in_chans=1)
    ck = torch.load(BB_USE, map_location='cpu', weights_only=False)
    miss = bb.load_state_dict({k: v for k, v in ck['sd'].items() if not k.startswith('head.')},
                              strict=False)
    bad = [k for k in miss.missing_keys if not k.startswith('head.')]
    assert not bad, f'thieu khoa khi nap backbone PadChest: {bad[:5]}'
    return bb, ck

_bb, _ck0 = build_backbone()
with torch.no_grad():
    _d = max((_IN_SD[k] - v).abs().max().item() for k, v in _bb.state_dict().items() if k in _IN_SD)
assert _d > 1e-4, (f'trong so GIONG HET ImageNet (lech {_d:.1e}) -> `convnextv2_padchest.pt` khong '
                   'duoc nap. Moi so ve do manh/corr se vo nghia.')
# cong 2: hinh dang feature map — KHONG tin gia dinh ve channels-first/last
_bb = _bb.to(dev).eval()
with torch.no_grad():
    _fm = _bb.forward_features(torch.zeros(2, 1, SIZE, SIZE, device=dev))
_shape = tuple(_fm.shape)
_CHLAST = _shape[-1] != _shape[-2]                # [B,H,W,Cf] neu chieu cuoi khac chieu ke cuoi
D_CXR = _shape[-1] if _CHLAST else _shape[1]
N_TOK = (_shape[1]*_shape[2]) if _CHLAST else (_shape[2]*_shape[3])
print(f'cong 2: feature map {_shape} -> {"channels-LAST" if _CHLAST else "channels-FIRST"} | '
      f'{N_TOK} token, d={D_CXR}')
assert 100 <= N_TOK <= 2000 and 256 <= D_CXR <= 2048, f'hinh dang bat thuong: {_shape}'
def to_tokens(fm):
    '''[B,H,W,Cf] hoac [B,Cf,H,W] -> [B, N, Cf]. Quyet dinh bang HINH DANG do duoc o cong 2.'''
    return fm.flatten(1, 2) if _CHLAST else fm.flatten(2).transpose(1, 2)
print(f'cong 3: trong so PadChest da nap | lech ImageNet {_d:.3f} | '
      f'ep {_ck0.get("ep", "?")} mAP PadChest {_ck0.get("mAP", float("nan")):.4f}')
del _bb; torch.cuda.empty_cache()

cong 1: ap_cols khop sklearn


model.safetensors: reconstructing file:   0%|          |  0.00B / 62.5MB            

model.safetensors: downloading bytes:           |  0.00B            

cong 2: feature map (2, 640, 31, 31) -> channels-FIRST | 961 token, d=640
cong 3: trong so PadChest da nap | lech ImageNet 1.077 | ep 1 mAP PadChest 0.2563


## C3 · Ba arm + cổng khoá trước

In [4]:
ARMS = {
  'cnn_full': dict(img=True, ehr=True),   # arm thang bac 2 — DOC LAP, backbone cua rieng no
}
def build_arms(seed=SEED):
    bb, _ = build_backbone(); bb = bb.to(dev)
    M, E, I = nn.ModuleDict(), nn.ModuleDict(), nn.ModuleDict()
    for tag, cfg in ARMS.items():
        torch.manual_seed(seed)                  # gieo lai TRUOC TUNG arm -> khoi tao giong nhau
        M[tag] = MLDecoder(C, D_CXR, D_MLD, d_ehr_in=D_ETOK, t1=cfg['ehr'])
        if cfg['ehr']: E[tag] = ColTokBag(FEAT.shape[1], D_ETOK, V=V_MIC)
        if cfg['img']: I[tag] = Tier1Img(D_CXR)
    return bb, M.to(dev), E.to(dev), I.to(dev)

BB, MOD, ETK, IMG = build_arms()
NP_ = {t: sum(p.numel() for m in ([MOD[t]] + [x[t] for x in (ETK, IMG) if t in x])
              for p in m.parameters() if p.requires_grad) for t in ARMS}
print(f'{"arm":<14}{"tham so dau doc":>16}')
for t in ARMS: print(f'  {t:<12}{NP_[t]:>16,}')
print(f'  backbone dung chung: {sum(p.numel() for p in BB.parameters()):,} (CO huan luyen)')

_p = torch.randn(4, N_TOK, D_CXR, device=dev); _pp = torch.randn(4, N_TOK, D_CXR, device=dev)
_h1 = torch.ones(4, device=dev); _h0 = torch.zeros(4, device=dev)
_e = torch.randn(4, FEAT.shape[1], device=dev)
_mi = torch.zeros(4, MIC_L, dtype=torch.long, device=dev); _mc = torch.zeros(4, MIC_L, device=dev)
assert not next(iter(MOD.values())).q.weight.requires_grad, 'query_embed dang hoc'
with torch.no_grad():
    for t in IMG:
        _rel = float((IMG[t](_p, _pp, _h1) - _p).norm() / (_p.norm() + 1e-9))
        assert _rel < 1e-3, f'{t}: Tier1Img nhieu {_rel:.2e} luc khoi tao, phai < 1e-3'
        assert (IMG[t](_p, _pp, _h0) - _p).abs().max() < 1e-6, f'{t}: has=0 KHONG cat duoc nhanh'
    m = MOD['cnn_full'].eval(); et = ETK['cnn_full'](_e, _mi, _mc)
    l1 = m(_p, et); _k = m.t1; m.t1 = None; l2 = m(_p, et); m.t1 = _k
    assert (l1 - l2).abs().max() < 1e-5, 'Tier1 KHONG zero-init'
print('cong 4: Tier1 zero-init | Tier1Img LayerScale + has=0 cat sach OK')
BB, MOD, ETK, IMG = build_arms()

arm            tham so dau doc
  cnn_full           6,362,664
  backbone dung chung: 14,980,240 (CO huan luyen)
cong 4: Tier1 zero-init | Tier1Img LayerScale + has=0 cat sach OK


## C4b · Chỉ **định nghĩa hàm** — bản rút gọn của C4, KHÔNG huấn luyện

Cắt nguyên văn phần đầu của C4: `DS`, `mk_loader`, `eval_loader`, `arm_forward`,
`predict`. Bỏ toàn bộ vòng huấn luyện và phần đọc nhật ký.

Dùng khi chỉ cần chạy cell suy luận TEST. **Chạy cell này thay cho C4.**


In [5]:
from torchvision import transforms as T2
crit = ASL(gn=4, gp=0).to(dev)
sig = lambda x: 1.0 / (1.0 + np.exp(-x))
MIDS_T, MCNT_T = torch.from_numpy(MIDS), torch.from_numpy(MCNT)
FEAT_T = torch.from_numpy(FEAT)
tf_tr = T2.Compose([T2.RandomResizedCrop(SIZE, scale=(0.85, 1.0), ratio=(0.95, 1.05),
                                         interpolation=T2.InterpolationMode.BICUBIC),
                    T2.RandomRotation(7)])
NEED_PREV = any(c['img'] for c in ARMS.values())

class DS(Dataset):
    def __init__(s, idx, train): s.idx, s.train = np.asarray(idx), train
    def __len__(s): return len(s.idx)
    def _img(s, i):
        im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg')).convert('L')
        if s.train: im = tf_tr(im)
        x = torch.from_numpy(np.asarray(im, dtype='uint8').copy()).unsqueeze(0).float().div_(255)
        return (x - MU) / SD
    def __getitem__(s, k):
        i = int(s.idx[k]); pr = int(PREV_ROW[i]); x = s._img(i)
        return dict(image=x, image_prev=(s._img(pr) if (NEED_PREV and pr >= 0)
                                         else torch.zeros_like(x)),
                    has=torch.tensor(1.0 if pr >= 0 else 0.0), row=i, pos=k,
                    y=torch.from_numpy(YM[i]))

def mk_loader(idx, train, bs, nw=NW):
    return DataLoader(DS(idx, train), batch_size=bs, shuffle=train, num_workers=nw,
                      pin_memory=True, drop_last=train, persistent_workers=nw > 0,
                      prefetch_factor=PREFETCH if nw > 0 else None)
_DLC = {}
def eval_loader(idx, bs):
    '''Tao MOT LAN roi dung lai — tao moi moi epoch se ngap spam teardown va ton thoi gian.'''
    k = (len(idx), int(idx[0]), int(idx[-1]), bs)
    if k not in _DLC: _DLC[k] = mk_loader(idx, False, bs)
    return _DLC[k]

def arm_forward(t, p, pp, row, has):
    cfg = ARMS[t]
    q = IMG[t](p, pp, has) if cfg['img'] else p
    et = (ETK[t](FEAT_T[row].to(dev), MIDS_T[row].to(dev), MCNT_T[row].to(dev))
          if cfg['ehr'] else None)
    return MOD[t](q, et)

@torch.no_grad()
def predict(idx, bs=None):
    bs = BS_EVAL if bs is None else bs      # 128 o 518 se TRAN RAM o 1022
    BB.eval(); MOD.eval(); ETK.eval(); IMG.eval()
    P = {t: np.zeros((len(idx), C), 'float32') for t in ARMS}
    for b in tqdm(eval_loader(idx, bs), desc='eval', leave=False):
        with torch.autocast('cuda', dtype=AMP, enabled=dev == 'cuda'):
            p = to_tokens(BB.forward_features(b['image'].to(dev, non_blocking=True))).float()
            pp = (to_tokens(BB.forward_features(b['image_prev'].to(dev, non_blocking=True))).float()
                  if NEED_PREV else p)
        hs = b['has'].to(dev)
        for t in ARMS:
            P[t][b['pos'].numpy()] = torch.sigmoid(
                arm_forward(t, p, pp, b['row'], hs).float()).cpu().numpy()
    assert not any(np.isnan(v).any() for v in P.values()), 'predict cho ra NaN'
    return P

print('C4b: da dinh nghia ' + ', '.join(['DS', 'mk_loader', 'eval_loader', 'arm_forward', 'predict']) + ' — KHONG huan luyen. Chay tiep cell suy luan TEST.')


C4b: da dinh nghia DS, mk_loader, eval_loader, arm_forward, predict — KHONG huan luyen. Chay tiep cell suy luan TEST.


## C5 · `0a` — vòng nén: bộ nén có giữ được bệnh lý không

Đo **duy nhất một biến**: ảnh vào đầu đọc là ảnh **thật** hay ảnh đã qua `decode(encode(·))`.
Checkpoint, EHR, phim trước, thứ tự — giữ nguyên từng bit.

```
AP_goc   : anh that tu shard
AP_<ten> : chinh anh do, sau vong nen cua ung vien <ten>
doc      : delta THEO TUNG NHAN, khong phai trung binh
```

**Phim trước luôn là ảnh THẬT.** Đúng như lúc triển khai: ảnh sinh thay ảnh chính, còn phim trước
thì kế thừa nguyên từ bệnh nhân gốc.

**Lát val phân tầng.** Lấy ngẫu nhiên 3.000 ảnh val thì nhãn hiếm nhất chỉ còn `1–2` ca dương và
`AP` vô nghĩa. Nên lấy tối đa `K_POS` ca dương của **mỗi** nhãn rồi bù cho đủ. Hệ quả: tỉ lệ mắc
trên lát này **khác** val đầy đủ, nên `AP` tuyệt đối ở đây **không so được** với con số triển khai
— chỉ đọc **delta**, vì mọi ứng viên dùng đúng một lát.

⚠ `1022` không chia hết cho `8` (mà cũng không chia hết cho `4`). Đệm lên `1024` rồi cắt lại về
`1022` sau khi giải mã — viền letterbox vốn đã đen nên đệm không thêm thông tin gì.

In [6]:
# ==== 0a · VONG NEN — bo nen co giu duoc benh ly khong ====
from PIL import Image

SEED_CK, EP_CK = 42, 8            # checkpoint dung de cham; doi o day neu muon seed khac

# ---------- ap_cols: ban trong du an, co cong doi chieu sklearn ----------
def ap_cols(Yc, Sc):
    o = np.full(Yc.shape[1], np.nan)
    for j in range(Yc.shape[1]):
        y, s = Yc[:, j], Sc[:, j]; n1 = y.sum()
        if n1 == 0 or n1 == len(y): continue
        r = np.argsort(-s, kind='stable'); yy, ss = y[r], s[r]
        tp, fp = np.cumsum(yy), np.cumsum(1 - yy)
        last = np.r_[np.diff(ss) != 0, True]; tp, fp = tp[last], fp[last]
        pr, rc = tp / (tp + fp), tp / n1
        o[j] = float((np.diff(np.r_[0.0, rc]) * pr).sum())
    return o
try:
    from sklearn.metrics import average_precision_score as _aps
    _r6 = np.random.default_rng(6); _y6 = (_r6.random((3000, C)) < 0.2).astype('float32')
    _s6 = _r6.random((3000, C)).astype('float32')
    assert np.nanmax(np.abs(ap_cols(_y6, _s6) - np.array(
        [_aps(_y6[:, j], _s6[:, j]) for j in range(C)]))) < 1e-6
    print('cong 1: ap_cols khop sklearn')
except ImportError:
    print('cong 1: BO QUA (khong co sklearn)')

# ---------- HET val. Khong co lat con nao. ----------
# Ban dau toi lay lat phan tang 3.000 anh de tiet kiem. Do la mot TAP THU TU, va no
# cat nhan o dung K_POS=40 — trong do co Round(ed) Atelectasis, chinh la nhan nhay
# -0.0268 giua hai luot. Gio lay het val.
# ⚠ Nhan hiem NHAT khong loi gi: Clavicle Fracture co 19 ca duong vi VAL CHI CO 19.
#   Do la tran cua val, khong phai tran cua cach lay mau. AP cua no van se nhieu.
IDX0A = I_VA
Y0A = YM[IDX0A]
N_POS = Y0A.sum(0).astype(int)
print(f'0a cham tren HET val: {len(IDX0A):,} anh '
      '-> AP tuyet doi so duoc voi cac so val khac cua du an')
print(f'  duong tinh moi nhan: min {N_POS.min()} | trung vi {int(np.median(N_POS))} '
      f'| max {N_POS.max()}')
_yeu = [LAB40[j] for j in np.argsort(N_POS)[:5]]
print('  5 nhan it duong tinh nhat: '
      + ', '.join(f'{n}({N_POS[LAB40.index(n)]})' for n in _yeu))
print(f'  uoc dia cho anh nen: ~{len(IDX0A) * 0.35e-3:.1f} GB moi ung vien '
      '(PNG 1022x1022 xam ~350 KB). Kiem `!df -h /content` neu lo.')
assert N_POS.min() >= 5, 'co nhan duoi 5 ca duong tren val -> AP cua nhan do la nhieu thuan'

# ---------- DS doc anh chinh tu IMG_DIR, phim truoc LUON tu shard that ----------
IMG_DIR, IMG_EXT = LOCAL, 'jpg'          # bien TOAN CUC — doi giua cac ung vien

class DS(Dataset):
    def __init__(s, idx, train): s.idx, s.train = np.asarray(idx), bool(train)
    def __len__(s): return len(s.idx)
    def _load(s, i, thu_muc, duoi):
        im = Image.open(os.path.join(thu_muc, f'{ORDER.dicom_id.iloc[i]}.{duoi}')).convert('L')
        x = torch.from_numpy(np.asarray(im, dtype='uint8').copy()).unsqueeze(0).float().div_(255)
        return (x - MU) / SD
    def __getitem__(s, k):
        i = int(s.idx[k]); pr = int(PREV_ROW[i])
        x = s._load(i, IMG_DIR, IMG_EXT)                       # anh CHINH — co the qua VAE
        xp = (s._load(pr, LOCAL, 'jpg') if (NEED_PREV and pr >= 0)   # phim truoc — LUON THAT
              else torch.zeros_like(x))
        return dict(image=x, image_prev=xp,
                    has=torch.tensor(1.0 if pr >= 0 else 0.0), row=i, pos=k,
                    y=torch.from_numpy(YM[i]))

# ---------- nap checkpoint cua 68b (chi DOC) ----------
_w = find(f'w_s{SEED_CK}_ep{EP_CK}.pt')
assert _w, (f'khong thay w_s{SEED_CK}_ep{EP_CK}.pt trong {len(DIRS_68B)} thu muc 68b. '
            'Kiem thu muc trung ten tren Drive.')
_wp = max(_w, key=os.path.getsize)
BB, MOD, ETK, IMG = build_arms(SEED_CK)
_ck = torch.load(_wp, map_location='cpu', weights_only=False)
assert int(_ck['seed']) == SEED_CK and int(_ck['ep']) == EP_CK - 1, 'file ghi seed/ep khac ten'
BB.load_state_dict(_ck['bb']); MOD.load_state_dict(_ck['mod'])
ETK.load_state_dict(_ck['etk']); IMG.load_state_dict(_ck['img']); del _ck
print(f'checkpoint: {os.path.basename(_wp)} tu {os.path.dirname(_wp).replace(OUT_DIR, "...")}')


def cham(thu_muc, duoi):
    """Doi nguon anh CHINH roi chay dung `predict` cua 68b."""
    global IMG_DIR, IMG_EXT
    IMG_DIR, IMG_EXT = thu_muc, duoi
    # eval_loader khoa theo (len, dau, cuoi, bs) — GIONG NHAU cho moi ung vien.
    # Khong xoa thi no dung lai loader cu (worker da fork, giu IMG_DIR cu) va ta
    # se do NHAM ANH GOC ba lan ma khong co dau hieu gi.
    for _k in list(_DLC): _DLC.pop(_k)
    _f = os.path.join(thu_muc, f'{ORDER.dicom_id.iloc[int(IDX0A[0])]}.{duoi}')
    assert os.path.exists(_f), f'DS se doc {_f} — khong co file do'
    return predict(IDX0A)['cnn_full']


# ---------- TAT DINH: bat buoc, neu khong thi phep do khong tai lap duoc ----------
# 15/09 do duoc: cung file anh, cung checkpoint, 9/40 nhan doi gia tri giua hai luot,
# lon nhat -0.0268 (Round(ed) Atelectasis, n+=40) — LON HON phan lon delta ma ta doc
# la "tin hieu". Va no chi danh vao nhan AP THAP: o do nhieu diem gan bang nhau nen
# nhieu rat nho cung lat thu hang. Ma nhan AP thap chinh la medium/tail ta quan tam.
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.use_deterministic_algorithms(True, warn_only=True)

# ---------- goc ----------
P_GOC = cham(LOCAL, 'jpg')
AP_GOC = ap_cols(Y0A, P_GOC)
print(f'\nGOC  macro {np.nanmean(AP_GOC):.4f} | '
      + ' '.join(f'{t} {np.nanmean(AP_GOC[i]):.4f}' for t, i in TIER.items()))

# ---------- cong 3: cham LAI dung tap anh do, AP phai KHONG DOI ----------
# Ton them mot luot predict (~2 phut). Khong co cong nay thi khong phan biet duoc
# "bo nen lam mat" voi "suy luan bat dinh" — 15/09 da do duoc 9/40 nhan doi gia tri
# tren DUNG CUNG file anh, lon nhat -0.0268 (Round(ed) Atelectasis, n+=40).
#
# CHAN theo AP chu khong theo xac suat tho: thu ta doc la AP, ma AP tinh theo THU
# HANG. Lech 1e-7 o xac suat thuong khong doi thu hang (vo hai), nhung o nhan AP
# thap — noi nhieu diem gan bang nhau — thi lech nho VAN lat thu hang (chet nguoi).
_P2 = cham(LOCAL, 'jpg')
_AP2 = ap_cols(Y0A, _P2)
_lech_p = float(np.abs(_P2 - P_GOC).max())
_lech_ap = float(np.nanmax(np.abs(_AP2 - AP_GOC)))
print(f'cong 3: lech xac suat lon nhat {_lech_p:.2e} | lech AP lon nhat {_lech_ap:.2e}')
if _lech_p > 0 and _lech_ap == 0:
    print('  (xac suat co jitter nhung THU HANG khong doi -> AP van tai lap duoc, chap nhan)')
assert _lech_ap == 0.0, (
    f'cham lai CUNG tap anh ra AP KHAC (lech lon nhat {_lech_ap:.2e} tren nhan '
    f'{LAB40[int(np.nanargmax(np.abs(_AP2 - AP_GOC)))]}).\n'
    '  Suy luan con bat dinh -> moi delta trong bang deu khong doc duoc.\n'
    '  Thu: dat AMP = torch.float32 o C0 roi chay lai C5 (cham hon nhung tat dinh),\n'
    '  hoac bo warn_only=True trong use_deterministic_algorithms de biet op nao gay ra.')
print('cong 3: AP tai lap duoc')

# ---------- cong: PNG khong lam doi mot pixel nao ----------
_d = '/content/rt/_gate'; os.makedirs(_d, exist_ok=True)
for _i in IDX0A[:8]:
    _im = Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[_i]}.jpg')).convert('L')
    _im.save(os.path.join(_d, f'{ORDER.dicom_id.iloc[_i]}.png'))
    _a = np.asarray(_im)
    _b = np.asarray(Image.open(os.path.join(_d, f'{ORDER.dicom_id.iloc[_i]}.png')).convert('L'))
    assert np.array_equal(_a, _b), 'PNG lam doi pixel -> phep so se ban'
print('cong 2: PNG khong lam doi mot pixel nao (8 anh)')


cong 1: ap_cols khop sklearn
lat 0a: 3,000 anh val | duong tinh moi nhan: min 19 | trung vi 86 | max 1050
  5 nhan it duong tinh nhat: Clavicle Fracture(19), Azygos Lobe(20), Lobar Atelectasis(26), Round(ed) Atelectasis(40), Pneumoperitoneum(42)
checkpoint: w_s42_ep8.pt tu .../stage4_nb68b/r1022_E8


eval:   0%|          | 0/94 [00:00<?, ?it/s]


GOC  macro 0.3094 | head 0.5635 medium 0.2694 tail 0.2040
cong 2: PNG khong lam doi mot pixel nao (8 anh)


## C6 · Vòng nén cho từng ứng viên → bảng `40 nhãn × ứng viên`

Ba ứng viên, và **`SDXL-VAE` gốc là đường cơ sở** chứ không phải thứ phải loại:

| Ứng viên | Latent @1024 | mm/ô | Vai trò |
|---|---|---|---|
| `sdxl` — `stabilityai/sdxl-vae` | `128×128×4` | `3,29` | **đường cơ sở** |
| `medvae_f64` — stride 8, C=4 | `128×128×4` | `3,29` | ứng viên, trùng hình dạng latent |
| `medvae_f16` — stride 4, C=3 | `256×256×3` | `1,64` | **chỉ chẩn đoán**, không triển khai vòng này |

`medvae_f16` chạy ở đây để trả lời *"mất mát do nén quá thô, hay do bản thân bộ nén"*. Nếu nó cứu
được thì kết luận là **thô quá** — và cách rẻ để bớt thô là cắt vùng, ghi ở mục đề xuất vòng sau,
**không phải** đổi sang `f=16` ở vòng này (đổi hình dạng latent là bỏ tiên nghiệm SDXL, `~80h`).

⚠ Ứng viên nào không nạp được thì cell **báo rồi đi tiếp**, không làm hỏng cả lượt. `diffusers` là
chuẩn nên `sdxl` gần như chắc chạy; `MedVAE` có thể cần đúng tên gói/API — chạy xong báo tôi lỗi
nguyên văn, tôi sửa.

**Đọc bảng:** cột `Δ` là `AP_ứng_viên − AP_gốc` **theo từng nhãn**. Số âm lớn ở một nhãn nghĩa là
bộ nén xoá mất đặc trưng của đúng nhãn đó.

In [7]:
# ==== C6 · vong nen cho tung ung vien ====
import subprocess, sys as _sy

BS_VAE = 4                      # ha xuong 2 neu tran VRAM o 1024
DIR_RT = '/content/rt'
os.makedirs(DIR_RT, exist_ok=True)


def _pad1024(x):
    """1022 khong chia het cho 8 (cung khong chia het cho 4). Dem doi xung len 1024.
       Vien letterbox von da den nen dem khong them thong tin gi."""
    h, w = x.shape[-2:]
    ph, pw = 1024 - h, 1024 - w
    assert 0 <= ph < 8 and 0 <= pw < 8, f'kich thuoc la: {h}x{w}'
    return F.pad(x, (pw // 2, pw - pw // 2, ph // 2, ph - ph // 2)), (ph, pw)


def _unpad(x, phw):
    ph, pw = phw
    h, w = x.shape[-2:]
    return x[..., ph // 2:h - (ph - ph // 2), pw // 2:w - (pw - pw // 2)]


def _giai_ma(vae, xp):
    """encode -> decode, chiu duoc ca hai kieu tra ve:
       - diffusers: doi tuong co .latent_dist (phan bo)
       - MedVAE   : Tensor THUAN

       DUNG isinstance, KHONG dung hasattr. Ly do (15/09 da vap): torch.Tensor CO
       phuong thuc .mode() — do la mode THONG KE, tra ve named tuple (values, indices).
       `hasattr(tensor, 'mode')` vi vay LUON True, va chuoi fallback theo hasattr se
       tong named tuple vao conv2d:
         TypeError: conv2d() ... got (torch.return_types.mode, Parameter, ...)"""
    e = vae.encode(xp)
    if isinstance(e, torch.Tensor):
        z = e                                    # MedVAE
    elif hasattr(e, 'latent_dist'):
        z = e.latent_dist.mode()                 # diffusers — mode, khong sample, de tat dinh
    elif isinstance(e, (tuple, list)):
        z = e[0]
    else:
        raise TypeError(f'encode() tra ve kieu la: {type(e)}')
    r = vae.decode(z)
    if isinstance(r, torch.Tensor):
        return r
    if hasattr(r, 'sample'):
        return r.sample
    if isinstance(r, (tuple, list)):
        return r[0]
    raise TypeError(f'decode() tra ve kieu la: {type(r)}')


@torch.no_grad()
def vong_nen(ten, vae, idx):
    """Ghi ban decode(encode(.)) ra PNG. Tra ve thu muc."""
    d = os.path.join(DIR_RT, ten); os.makedirs(d, exist_ok=True)
    # Chu ky ghi BO NEN + TIEN XU LY, KHONG ghi tham so lat.
    # (Sua 15/09: ban dau toi ghi tham so lat — SAI BAN CHAT. Anh PNG cua mot dicom_id
    #  khong phu thuoc vao lat, chi phu thuoc ANH GOC va BO NEN. Ghi tham so lat lam
    #  anh da nen bi vut di moi khi doi pham vi, hoan toan vo co.)
    _ky = dict(ung_vien=ten, size=int(SIZE), dem_len=1024,
               kenh_vao=3, kenh_ra='trung binh', lay_z='mode')
    _pk = os.path.join(d, '_bo_nen.json')
    if os.path.exists(_pk):
        _cu = json.load(open(_pk))
        assert _cu == _ky, (
            f'{ten}: thu muc dang giu anh nen bang CAU HINH KHAC.\n'
            f'  da co : {_cu}\n  bay gio: {_ky}\n'
            f'  Xoa TAY {d} roi chay lai. Cell nay KHONG tu xoa gi.')
    else:
        json.dump(_ky, open(_pk, 'w'))
    # Thu muc co the dang giu anh cua luot truoc voi pham vi NHO HON — do la binh thuong
    # va DUNG: chung dung lai duoc, chi can nen them phan con thieu.
    vae = vae.to(dev).eval()
    for fn in ('enable_slicing', 'enable_tiling'):
        if hasattr(vae, fn):
            try: getattr(vae, fn)()
            except Exception: pass
    con = [int(i) for i in idx
           if not os.path.exists(os.path.join(d, f'{ORDER.dicom_id.iloc[int(i)]}.png'))]
    print(f'  {ten}: can nen {len(con):,}/{len(idx):,} anh')
    bs = BS_VAE
    k = 0
    with tqdm(total=len(con), desc=ten, leave=False) as thanh:
        while k < len(con):
            lo = con[k:k + bs]
            ims = [np.asarray(Image.open(os.path.join(LOCAL, f'{ORDER.dicom_id.iloc[i]}.jpg'))
                              .convert('L'), dtype='uint8') for i in lo]
            x = torch.from_numpy(np.stack(ims)).unsqueeze(1).float().div_(255.)
            x = (x * 2 - 1).repeat(1, 3, 1, 1).to(dev)     # VAE nhan [-1,1], 3 kenh
            xp, phw = _pad1024(x)
            try:
                with torch.autocast('cuda', dtype=AMP):
                    xr = _giai_ma(vae, xp)
            except torch.cuda.OutOfMemoryError:
                # MedVAE stride 4 co attention vanilla o 256x256 = 65.536 token -> O(n^2).
                # bs=4 doi 64 GiB. Lui dan thay vi hong ca luot.
                del x, xp; torch.cuda.empty_cache()
                assert bs > 1, f'{ten}: OOM ngay ca voi bs=1 -> khong do duoc o {SIZE}'
                bs = max(1, bs // 2); print(f'  {ten}: OOM -> lui batch xuong {bs}')
                continue
            assert torch.isfinite(xr).all(), (
                f'{ten}: VAE ra NaN/Inf o lo {k}. AMP dang la fp16 thi doi sang bf16, '
                'hoac chay VAE o float32.')
            xr = _unpad(xr.float(), phw)
            xr = ((xr / 2 + 0.5).clamp(0, 1).mean(1) * 255).round().to(torch.uint8).cpu().numpy()
            for i, a in zip(lo, xr):
                Image.fromarray(a).save(os.path.join(d, f'{ORDER.dicom_id.iloc[i]}.png'))
            k += len(lo); thanh.update(len(lo))
    _thieu = [i for i in idx
              if not os.path.exists(os.path.join(d, f'{ORDER.dicom_id.iloc[int(i)]}.png'))]
    assert not _thieu, f'{ten}: con thieu {len(_thieu):,}/{len(idx):,} anh'
    del vae; torch.cuda.empty_cache()
    return d


# ---------- dang ky ung vien ----------
def _tai_sdxl():
    try:
        from diffusers import AutoencoderKL
    except ImportError:
        subprocess.run([_sy.executable, '-m', 'pip', 'install', '-q', 'diffusers'])
        from diffusers import AutoencoderKL
    return AutoencoderKL.from_pretrained('stabilityai/sdxl-vae')


def _tai_medvae(ten_mo_hinh):
    """MedVAE (arXiv:2502.14753, StanfordMIMI/MedVAE). API co the doi —
       thu vai loi vao roi bao loi NGUYEN VAN neu khong duoc."""
    try:
        import medvae
    except ImportError:
        subprocess.run([_sy.executable, '-m', 'pip', 'install', '-q', 'medvae'])
        import medvae
    for duong in (
        lambda: medvae.MVAE(model_name=ten_mo_hinh, modality='xray'),
        lambda: medvae.MVAE(model_name=ten_mo_hinh),
        lambda: medvae.utils.factory.create_model(ten_mo_hinh),
    ):
        try:
            return duong()
        except Exception as e:
            _cuoi = e
    raise RuntimeError(f'khong dung duoc loi vao nao cua medvae: {type(_cuoi).__name__}: {_cuoi}')


# f16 la CHAN DOAN, khong phai phuong an trien khai. Muc dich cua no la tra loi
# "mat mat do nen qua tho hay do ban than bo nen" — nhung lan chay 15/09 cho thay
# SDXL-VAE goc chi lam hong <=2/31 nhan, tuc gan nhu khong co mat mat de giai thich.
# Va no cham: attention vanilla o 256x256 = 65.536 token, phai lui ve bs=1.
CHAY_F16 = False

UNG_VIEN = [
    ('sdxl',        'duong co so — stride 8, C=4',            _tai_sdxl),
    ('medvae_f64',  'ung vien  — stride 8, C=4, trung hinh dang',
     lambda: _tai_medvae('medvae_8_4_2d')),
]
if CHAY_F16:
    UNG_VIEN.append(('medvae_f16', 'CHAN DOAN — stride 4, C=3, KHAC hinh dang',
                     lambda: _tai_medvae('medvae_4_3_2d')))

AP = {'goc': AP_GOC}
HONG = {}
for ten, mo_ta, tai in UNG_VIEN:
    print(f'\n--- {ten}  ({mo_ta}) ---')
    # Tach NAP khoi NEN: gop chung thi loi o vong nen cung bi gan nhan 'khong nap duoc',
    # ma do la hai cho hong hoan toan khac nhau (15/09 da lam roi).
    try:
        _vae = tai()
    except Exception as e:
        HONG[ten] = f'NAP HONG — {type(e).__name__}: {e}'
        print(f'  {HONG[ten]}'); continue
    try:
        d = vong_nen(ten, _vae, IDX0A)
    except Exception as e:
        HONG[ten] = f'NAP DUOC nhung VONG NEN HONG — {type(e).__name__}: {e}'
        print(f'  {HONG[ten]}'); torch.cuda.empty_cache(); continue
    _P = cham(d, 'png')
    assert float(np.abs(_P - P_GOC).mean()) > 1e-6, (
        f'{ten}: du doan TRUNG KHIT ban goc -> loader cu bi dung lai, anh qua VAE '
        'khong he duoc doc. Day la loi IM LANG: bang se ra delta = 0 o moi nhan.')
    AP[ten] = ap_cols(Y0A, _P)
    _dl = AP[ten] - AP_GOC
    print(f'  macro {np.nanmean(AP[ten]):.4f} (goc {np.nanmean(AP_GOC):.4f}, '
          f'delta {np.nanmean(_dl):+.4f}) | '
          + ' '.join(f'{t} {np.nanmean(_dl[i]):+.4f}' for t, i in TIER.items()))

# ---------- bang theo tung nhan ----------
_ten = [t for t, _, _ in UNG_VIEN if t in AP]
_tier_cua = {j: t for t, ii in TIER.items() for j in ii}
print('\n' + '=' * (34 + 18 * len(_ten)))
print(f'{"nhan":<26}{"tier":<7}{"n+":>5}  ' + '  '.join(f'{t:>16}' for t in _ten))
print('-' * (34 + 18 * len(_ten)))
for j in np.argsort(N_POS):
    print(f'{LAB40[j][:25]:<26}{_tier_cua[j][:6]:<7}{N_POS[j]:>5}  '
          + '  '.join(f'{AP[t][j]:>7.4f}{AP[t][j]-AP_GOC[j]:>+9.4f}' for t in _ten))
print('-' * (34 + 18 * len(_ten)))
print(f'{"MACRO":<26}{"":<7}{"":>5}  '
      + '  '.join(f'{np.nanmean(AP[t]):>7.4f}{np.nanmean(AP[t]-AP_GOC):>+9.4f}' for t in _ten))

# ---------- luu ----------
_kq = dict(ngay=datetime.datetime.now().isoformat(timespec='seconds'),
           checkpoint=os.path.basename(_wp), n_anh=int(len(IDX0A)),
           pham_vi='HET val', nhan=LAB40,
           duong_tinh_tren_lat=[int(x) for x in N_POS],
           tier={t: [int(x) for x in i] for t, i in TIER.items()},
           ap={t: [None if np.isnan(v) else float(v) for v in AP[t]] for t in AP},
           khong_nap_duoc=HONG)
# Ten co dau thoi gian: chay lai 0a (rat de xay ra neu phai sua API MedVAE) thi
# KHONG ghi de ban cu. Ban cu la ho so, khong phai rac.
_TEN_KQ = f'0a_vong_nen_{datetime.datetime.now():%Y%m%d_%H%M}.json'
put(_TEN_KQ, lambda p: json.dump(_kq, open(p, 'w'), ensure_ascii=False, indent=1))
print(f'\n{_TEN_KQ}: ' + ('THAT BAI, chi con ban local' if _TEN_KQ in PUT_FAIL
                          else f'da luu -> {RES41D.replace(OUT_DIR, "...")}'))

# ---------- SAN NHIEU: bo nen KHONG THE lam TANG AP ----------
# Nen moi delta DUONG la nhieu thuan tuy, va nua duong cua phan bo la mot phep do
# TRUC TIEP cua san nhieu. Khong co no thi khong doc noi mot delta am la that hay nhieu.
for t in _ten:
    _dl = AP[t] - AP_GOC
    _d = _dl[np.isfinite(_dl)]
    _duong = _d[_d > 0]
    if len(_duong) < 3: continue
    _sig = float(np.sqrt((_duong ** 2).mean())); _ng = 2 * _sig
    print(f'\n--- {t}: san nhieu uoc tu {len(_duong)} nhan co delta duong ---')
    print(f'  rms {_sig:.4f} -> nguong 2*sigma = {_ng:.4f}')
    _vuot = [j for j in range(C) if np.isfinite(_dl[j]) and abs(_dl[j]) > _ng]
    if not _vuot:
        print('  KHONG nhan nao vuot san nhieu -> bo nen nay khong lam mat gi do duoc')
    for j in sorted(_vuot, key=lambda x: _dl[x]):
        print(f'  {LAB40[j][:26]:<27}n+={N_POS[j]:<5}delta {_dl[j]:+.4f}'
              + ('  <- MAT' if _dl[j] < 0 else '  <- TANG, ma tang la KHONG THE => nguong con lot nhieu'))
    _nho = np.array([abs(_dl[j]) for j in range(C) if N_POS[j] <= 70 and np.isfinite(_dl[j])])
    _lon = np.array([abs(_dl[j]) for j in range(C) if N_POS[j] > 200 and np.isfinite(_dl[j])])
    print(f'  |delta| tb: n+<=70 {_nho.mean():.4f} ({len(_nho)} nhan) | '
          f'n+>200 {_lon.mean():.4f} ({len(_lon)} nhan)')
    print('  -> lech lon o nhan it duong tinh la chu ky cua NHIEU UOC LUONG, khong phai')
    print('     chu ky cua bo nen (bo nen xoa cau truc manh thi loi phai theo KICH THUOC')
    print('     ton thuong, khong theo CO MAU).')

# ---------- bao cao TRUOC, chan SAU ----------
# Thu tu quan trong: neu cong Drive nem loi truoc khi in HONG thi mat luon thong tin
# can de sua API bo nen.
if HONG:
    print('\nUNG VIEN KHONG NAP DUOC — gui nguyen van loi nay de sua API:')
    for t, e in HONG.items(): print(f'  {t}: {e}')

# Cong Drive viet THANG o day, khong goi drive_gate() cua C0: mot ham 4 dong khong
# dang tao phu thuoc xuyen cell — chay lai MOT MINH C6 se NameError (da vap 15/09).
if PUT_FAIL:
    raise AssertionError(
        f'{len(PUT_FAIL)} file KHONG len duoc Drive: {PUT_FAIL[:4]}\n'
        f'  Hien vat CON o {RES_LOC} — dung tat session truoc khi cuu.\n'
        f'  Cuu ngay: !cp -r {RES_LOC} /content/drive/MyDrive/_cuu70a\n'
        f'  Roi CHAY LAI C0 (ban va 15/09 giu lai file danh dau nen thu muc khong\n'
        f'  bien mat nua, va put() co retry 3 lan) truoc khi chay lai C6.')
print('\ncong Drive: tat ca file da len' if not PUT_FAIL else '')



--- sdxl  (duong co so — stride 8, C=4) ---


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


config.json:   0%|          | 0.00/607 [00:00<?, ?B/s]

diffusion_pytorch_model.safetensors: reconstructing file:   0%|          |  0.00B /  335MB            

diffusion_pytorch_model.safetensors: downloading bytes:           |  0.00B            

...

model_weights/vae_4x_3c_2D.ckpt: reconstructing file:   0%|          |  0.00B /  224MB            

model_weights/vae_4x_3c_2D.ckpt: downloading bytes:           |  0.00B            

making attention of type 'vanilla' with 512 in_channels
Working with z of shape (1, 3, 64, 64) = 12288 dimensions.
making attention of type 'vanilla' with 512 in_channels
Restored from /root/.cache/huggingface/hub/models--stanfordmimi--MedVAE/snapshots/8040c5ccb7a51eff6b4cef513c7f20bd1bda983d/model_weights/vae_4x_3c_2D.ckpt with 0 missing and 0 unexpected keys
  medvae_f16: can nen 3,000/3,000 anh


medvae_f16:   0%|          | 0/750 [00:00<?, ?it/s]

  KHONG NAP DUOC -> bo qua ung vien nay
  OutOfMemoryError: CUDA out of memory. Tried to allocate 64.00 GiB. GPU 0 has a total capacity of 94.97 GiB of which 54.33 GiB is free. Including non-PyTorch memory, this process has 40.63 GiB memory in use. Of the allocated memory 38.84 GiB is allocated by PyTorch, and 1.14 GiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

nhan                      tier      n+              sdxl
----------------------------------------------------
Clavicle Fracture         tail      19   0.0230  +0.0012
Azygos Lobe               tail      20   0.1051  +0.0213
Lobar Atelectasis         tail      26   0.0226  +0.0003
Round(ed) Atelectasis     tail      40   0.1197  -0.0044
Pneumoperitoneum     